In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv("qoute_dataset.csv")

In [7]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [8]:
df.shape

(3038, 2)

In [9]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


Lower Case

In [10]:
quotes = quotes.str.lower()

In [11]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,"“it is our choices, harry, that show what we t..."
2,“there are only two ways to live your life. on...
3,"“the person, be it gentleman or lady, who has ..."
4,"“imperfection is beauty, madness is genius and..."


Remove punctuation

In [12]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [13]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


Tokenizetion

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [15]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [16]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:15]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10),
 ('in', 11),
 ('be', 12),
 ('not', 13),
 ('are', 14),
 ('your', 15)]

In [17]:
sequence = tokinizer.texts_to_sequences(quotes)

In [18]:
for i in range(4):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”
“the person be it gentleman or lady who has not pleasure in a good novel must be intolerably stupid”


In [19]:
for i in range(4):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]
[713, 116, 12, 10, 2462, 32, 1043, 30, 82, 13, 601, 11, 5, 74, 1338, 119, 12, 2463, 3678]


In [20]:
print(tokinizer.word_index['“it'])

947


In [21]:
print(tokinizer.word_index['it'])

10


In [22]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [23]:
len(X)

85271

In [24]:
len(y)

85271

Padding

In [25]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [26]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [27]:
y = np.array(y)

In [28]:
X_padded.shape

(85271, 745)

In [29]:
y.shape

(85271,)

In [30]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [31]:
y

array([ 62,  29,  19, ...,   3, 169, 101])

One Hot Encodding

In [32]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [33]:
y_one_hot.shape

(85271, 10000)

In [34]:
y_one_hot

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN, LSTM, Dense,GRU

Embedding

In [36]:
embedding_dim = 50
rnn_units = 128

RNN Model

In [37]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [38]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

LSTM Model

In [40]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [41]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [42]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# GRU Model

In [43]:
gru_model = Sequential()

gru_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)

gru_model.add(GRU(units=rnn_units))
gru_model.add(Dense(units=vocab_size, activation='softmax'))

gru_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [44]:
epochs =100
batch_size = 128
history_gru = gru_model.fit(
    X_padded,
    y_one_hot,
    epochs=100,
    batch_size=128,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 36s 49ms/step - accuracy: 0.0438 - loss: 6.8047 - val_accuracy: 0.0580 - val_loss: 6.5442
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 28s 46ms/step - accuracy: 0.0813 - loss: 6.0875 - val_accuracy: 0.0960 - val_loss: 6.3243
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 28s 47ms/step - accuracy: 0.1062 - loss: 5.7299 - val_accuracy: 0.1060 - val_loss: 6.2726
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 28s 47ms/step - accuracy: 0.1214 - loss: 5.4391 - val_accuracy: 0.1120 - val_loss: 6.2891
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.1373 - loss: 5.1792 - val_accuracy: 0.1157 - val_loss: 6.3255
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.1512 - loss: 4.9387 - val_accuracy: 0.1173 - val_loss: 6.3599
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.1645 - loss: 4.7131 - val_accuracy: 0.1195 - val_loss: 6.4220
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 29s 49ms/step - accuracy: 0.1797 - loss: 4

In [45]:
gru_model.save('gru_model.h5')

In [46]:
gru_acc = history_gru.history['accuracy'][-1]
print("gru Accuracy:", gru_acc)

gru Accuracy: 0.7584275007247925
